In [52]:
%matplotlib inline
import joblib
import shap
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import set_config
from tempfile import TemporaryDirectory
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score
from fastparquet import write
from fastparquet import ParquetFile
from sklearn import metrics
import seaborn as sns
import numpy as np
import json
import os

In [53]:
experiment_path = "2026-08-22-08_54_45_PM_xgboost"

In [54]:
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    data_path = "/kaggle/input/datasets/abhinavneelam/smartphone-addiction/dataset/data"
    experiments_path = "/kaggle/input/datasets/abhinavneelam/smartphone-addiction/dataset/experiments"
    output_path = "/kaggle/working/"
else:
    data_path = "../../data"
    experiments_path = "../../experiments"
    output_path = "../../"

In [55]:
ss = pd.read_csv(f"{data_path}/raw/sample_submission.csv")
target_column = ss.columns[-1]
target_column

'addicted_label'

In [56]:
train_ft_groups, test_ft_groups = [], []
with open(f"{experiments_path}/{experiment_path}/full_config.json") as f:
    experiment_config = json.load(f)
    train_ft_groups = experiment_config["experiment"]["train_feature_groups"]
    test_ft_groups = experiment_config["experiment"]["test_feature_groups"]

model = joblib.load(f"{experiments_path}/{experiment_path}/{experiment_path.split('_')[-1]}.pkl")

In [57]:
raw_train_df = pd.read_csv(f"{data_path}/raw/train.csv")
raw_train_id = raw_train_df["id"]
X_dfs, X_test_dfs = [], []
baseline_features = []

for fg in train_ft_groups:
    file_path = f"{data_path}/{fg}"
    df = ParquetFile(file_path).to_pandas()
    if fg.startswith("baseline"):
        baseline_features = df.columns
    X_dfs.append(df)

for fg in test_ft_groups:
    file_path = f"{data_path}/{fg}"
    df = ParquetFile(file_path).to_pandas()
    X_test_dfs.append(df)

X = pd.concat(X_dfs, axis=1)
X_test = pd.concat(X_test_dfs, axis=1)
y = raw_train_df[target_column]

X.info()

<class 'pandas.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 40 columns):
 #   Column                            Non-Null Count   Dtype   
---  ------                            --------------   -----   
 0   age                               662440 non-null  float64 
 1   daily_screen_time_hours           595515 non-null  float64 
 2   social_media_hours                557374 non-null  float64 
 3   gaming_hours                      564548 non-null  float64 
 4   work_study_hours                  639851 non-null  float64 
 5   sleep_hours                       646889 non-null  float64 
 6   notifications_per_day             623785 non-null  float64 
 7   app_opens_per_day                 610659 non-null  float64 
 8   weekend_screen_time               579306 non-null  float64 
 9   gender                            662335 non-null  category
 10  stress_level                      636221 non-null  float64 
 11  academic_work_impact              647145 non-null 

**Group Permutation Importance**

In [ ]:
baseline_features = np.asarray(baseline_features, dtype=str)
cols = np.asarray(X.columns, dtype=str)
feature_groups = ["TE"]

In [59]:
def get_group_columns(cols, group):
    prefix = f"{group}_"
    suffix = f"_{group}"

    mask = (
        (cols == group)
        | np.char.startswith(cols, prefix)
        | np.char.endswith(cols, suffix)
    )

    return cols[mask].tolist()

In [60]:
def build_feature_groups(X_columns, baseline_features, feature_groups):
    X_columns = np.asarray(X_columns, dtype=str)
    baseline_features = np.asarray(baseline_features, dtype=str)

    # Keep only baseline columns that actually exist in X
    baseline_set = set(baseline_features)
    baseline_cols = [
        col for col in X_columns
        if col in baseline_set
    ]

    # Track columns already assigned to a bucket
    assigned = set(baseline_cols)

    groups = {
        "baseline": baseline_cols
    }

    for group in feature_groups:
        group_cols = get_group_columns(X_columns, group)

        group_cols = [
            col for col in group_cols
            if col not in assigned
        ]

        groups[group] = group_cols

        assigned.update(group_cols)

    # Extra engineered features
    extra_cols = [
        col for col in X_columns
        if col not in assigned
    ]

    groups["extra"] = extra_cols
    return groups

In [61]:
group_columns = build_feature_groups(
    X_columns=cols,
    baseline_features=baseline_features,
    feature_groups=feature_groups,
)

print("\nFeature groups:")

for group, group_cols in group_columns.items():
    print(f"{group:>10}: {len(group_cols):4d} features")

    if group_cols:
        print(f"            {group_cols}")

all_grouped_columns = [
    col
    for group_cols in group_columns.values()
    for col in group_cols
]

assert len(all_grouped_columns) == len(cols), (
    "Some columns were not assigned to a feature bucket."
)

assert len(set(all_grouped_columns)) == len(cols), (
    "A column was assigned to multiple feature buckets."
)

assert set(all_grouped_columns) == set(cols), (
    "Feature bucket assignment does not match X.columns."
)


Feature groups:
  baseline:   12 features
            [np.str_('age'), np.str_('daily_screen_time_hours'), np.str_('social_media_hours'), np.str_('gaming_hours'), np.str_('work_study_hours'), np.str_('sleep_hours'), np.str_('notifications_per_day'), np.str_('app_opens_per_day'), np.str_('weekend_screen_time'), np.str_('gender'), np.str_('stress_level'), np.str_('academic_work_impact')]
        TE:   12 features
            ['age_TE', 'daily_screen_time_hours_TE', 'social_media_hours_TE', 'gaming_hours_TE', 'work_study_hours_TE', 'sleep_hours_TE', 'notifications_per_day_TE', 'app_opens_per_day_TE', 'weekend_screen_time_TE', 'gender_TE', 'stress_level_TE', 'academic_work_impact_TE']
     extra:   16 features
            [np.str_('screen_sleep_ratio'), np.str_('total_activity'), np.str_('social_media_ratio'), np.str_('gaming_ratio'), np.str_('work_ratio'), np.str_('daily_free_hours'), np.str_('daily_extra_screen_time_hours'), np.str_('weekend_extra_screen_time'), np.str_('phone_activity'

In [62]:
def grouped_permutation_importance(
    model,
    X,
    y,
    group_columns,
    n_repeats=10,
    random_state=0,
):
    rng = np.random.default_rng(random_state)

    baseline_pred = model.predict_proba(X)[:, 1]
    baseline_auc = roc_auc_score(y, baseline_pred)

    results = []

    for group, group_cols in group_columns.items():
        if not group_cols:
            print(f"[SKIP] {group}: no features")
            continue

        print(
            f"Evaluating {group!r}: "
            f"{len(group_cols)} features"
        )

        permuted_aucs = []

        for repeat in range(n_repeats):
            permutation = rng.permutation(len(X))

            X_permuted = X.copy()
            X_permuted.loc[:, group_cols] = (
                X.iloc[permutation][group_cols].to_numpy()
            )
            predictions = model.predict_proba(X_permuted)[:, 1]
            auc = roc_auc_score(y, predictions)

            permuted_aucs.append(auc)

        permuted_aucs = np.asarray(permuted_aucs)
        importance = baseline_auc - permuted_aucs

        results.append(
            {
                "group": group,
                "n_features": len(group_cols),

                "baseline_auc": baseline_auc,

                "permuted_auc_mean": permuted_aucs.mean(),
                "permuted_auc_std": permuted_aucs.std(ddof=1),

                "importance_mean": importance.mean(),
                "importance_std": importance.std(ddof=1),

                "features": group_cols,
            }
        )

    results_df = pd.DataFrame(results)

    results_df = results_df.sort_values(
        "importance_mean",
        ascending=False,
    ).reset_index(drop=True)

    return results_df

In [63]:
importance_df = grouped_permutation_importance(
    model=model,
    X=X,
    y=y,
    group_columns=group_columns,
    n_repeats=10,
    random_state=0,
)

Evaluating 'baseline': 12 features
Evaluating 'TE': 12 features
Evaluating 'extra': 16 features


In [64]:
importance_df

,group,n_features,baseline_auc,permuted_auc_mean,permuted_auc_std,importance_mean,importance_std,features
0,TE,12,0.95447,0.675805,0.000560,0.278665,0.000560,"[age_TE, daily_screen_time_hours_TE, social_me..."
1,baseline,12,0.95447,0.930641,0.000111,0.023829,0.000111,"[age, daily_screen_time_hours, social_media_ho..."
2,extra,16,0.95447,0.946616,0.000075,0.007854,0.000075,"[screen_sleep_ratio, total_activity, social_me..."
